# SVD-Based Collaborative Filtering Model (Optimized)

A production-ready collaborative filtering recommender system using Singular Value Decomposition with optimal techniques including normalization, bias correction, and 600 latent factors.

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')


## 1. Load and Prepare Data

In [2]:
# Load the dataset
df = pd.read_csv('final_dataset.csv')
print("Dataset shape:", df.shape)
print("\nUnique users:", df['userid'].nunique())
print("Unique movies:", df['movieid'].nunique())
print("\nTotal ratings:", len(df))

# Create user-item rating matrix
rating_matrix = df.pivot_table(index='userid', columns='movieid', values='rating')
print(f"\nRating matrix shape: {rating_matrix.shape}")
print(f"Sparsity: {(rating_matrix.isna().sum().sum() / (rating_matrix.shape[0] * rating_matrix.shape[1])) * 100:.2f}%")

Dataset shape: (321026, 24)

Unique users: 610
Unique movies: 6834

Total ratings: 321026

Rating matrix shape: (610, 6834)
Sparsity: 98.26%


## 2. Train-Test Split

In [3]:

observed_ratings = df[['userid', 'movieid', 'rating']].copy()

# 80-20 train-test split
train_data, test_data = train_test_split(observed_ratings, test_size=0.2, random_state=42)

# training matrix
train_matrix = train_data.pivot_table(index='userid', columns='movieid', values='rating')
train_matrix_filled = train_matrix.fillna(0)


## 3. Normalization & Bias Correction

In [4]:
# Calculate global mean
global_mean = train_data['rating'].mean()


# Normalize by global mean
train_data_normalized = train_data.copy()
train_data_normalized['rating'] = train_data_normalized['rating'] - global_mean

# Calculate user and item biases
user_biases = train_data.groupby('userid')['rating'].mean() - global_mean
item_biases = train_data.groupby('movieid')['rating'].mean() - global_mean


# Double normalization: remove both user and item biases
train_data_doubly_norm = train_data_normalized.copy()
train_data_doubly_norm['rating'] = train_data_doubly_norm.apply(
    lambda row: row['rating'] - user_biases.get(row['userid'], 0) - item_biases.get(row['movieid'], 0),
    axis=1
)

# Create doubly normalized matrix
train_matrix_double = train_data_doubly_norm.pivot_table(index='userid', columns='movieid', values='rating')
train_matrix_double_filled = train_matrix_double.fillna(0)

print(f"\nDoubly normalized matrix shape: {train_matrix_double_filled.shape}")
print(f"Doubly normalized ratings mean: {train_data_doubly_norm['rating'].mean():.6f}")


Doubly normalized matrix shape: (610, 6532)
Doubly normalized ratings mean: -0.000000


## 4. SVD Model Training (600 Factors)

In [5]:
# Fit SVD with 600 latent factors (optimal configuration)
print("Training SVD model with 600 latent factors...")
svd_model = TruncatedSVD(n_components=600, random_state=42, n_iter=100)
U = svd_model.fit_transform(train_matrix_double_filled)
Vt = svd_model.components_


Training SVD model with 600 latent factors...


## 5. Model Evaluation on Test Set

In [6]:
# Reconstruct predictions
pred_latent = U @ Vt
pred_df = pd.DataFrame(pred_latent, index=train_matrix_double.index, columns=train_matrix_double.columns)

# Function to predict rating with bias correction
def predict_rating(user_id, movie_id):
    """
    Predict rating for a user-movie pair.
    Formula: prediction = global_mean + user_bias + item_bias + latent_prediction
    """
    if user_id not in pred_df.index or movie_id not in pred_df.columns:
        return None
    
    latent_pred = pred_df.loc[user_id, movie_id]
    user_bias = user_biases.get(user_id, 0)
    item_bias = item_biases.get(movie_id, 0)
    
    # Reconstruct full prediction
    full_pred = global_mean + user_bias + item_bias + latent_pred
    
    # Clip to valid rating range [0.5, 5.0]
    return np.clip(full_pred, 0.5, 5.0)

# Evaluate on test set
test_predictions = []
test_actuals = []

for _, row in test_data.iterrows():
    user_id = row['userid']
    movie_id = row['movieid']
    actual_rating = row['rating']
    
    pred = predict_rating(user_id, movie_id)
    if pred is not None:
        test_predictions.append(pred)
        test_actuals.append(actual_rating)

test_predictions = np.array(test_predictions)
test_actuals = np.array(test_actuals)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
mae = mean_absolute_error(test_actuals, test_predictions)


print(f"\nRMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")



RMSE: 0.3256
MAE: 0.0877


## 6. Recommendation Functions

In [7]:
def get_recommendations(user_id, n_recommendations=5):
    """
    Generate top-N movie recommendations for a given user.
    
    Parameters:
    - user_id: The ID of the user
    - n_recommendations: Number of recommendations to return
    
    Returns:
    - DataFrame with recommended movies and predicted ratings
    """
    if user_id not in train_matrix.index:
        return f"User {user_id} not found in the dataset"
    
    # Get movies already rated by user
    user_idx = train_matrix.index.get_loc(user_id)
    user_rated = train_matrix.iloc[user_idx][train_matrix.iloc[user_idx] > 0].index.tolist()
    
    # Get predicted ratings for all movies
    recommendations = []
    for movie_id in train_matrix.columns:
        if movie_id not in user_rated:
            pred_rating = predict_rating(user_id, movie_id)
            if pred_rating is not None:
                recommendations.append({'movieid': movie_id, 'predicted_rating': pred_rating})
    
    # Sort by predicted rating and get top N
    recommendations_df = pd.DataFrame(recommendations).nlargest(n_recommendations, 'predicted_rating')
    
    # Add movie titles
    rec_movies = df[df['movieid'].isin(recommendations_df['movieid'].values)][['movieid', 'clean_title', 'vote_average']].drop_duplicates()
    result = recommendations_df.merge(rec_movies, on='movieid', how='left')
    
    return result[['clean_title', 'predicted_rating', 'vote_average']].head(n_recommendations)

def find_similar_users(user_id, n_similar=5):
    """
    Find users with similar rating patterns based on latent factors.
    
    Parameters:
    - user_id: The ID of the target user
    - n_similar: Number of similar users to return
    
    Returns:
    - DataFrame with similar user IDs and similarity scores
    """
    if user_id not in train_matrix.index:
        return f"User {user_id} not found"
    
    user_idx = train_matrix.index.get_loc(user_id)
    user_vector = U[user_idx].reshape(1, -1)
    
    # Calculate cosine similarity with all users
    similarities = cosine_similarity(user_vector, U)[0]
    
    # Get top similar users (excluding the user itself)
    similar_indices = np.argsort(similarities)[::-1][1:n_similar+1]
    similar_users = train_matrix.index[similar_indices]
    similar_scores = similarities[similar_indices]
    
    return pd.DataFrame({
        'user_id': similar_users,
        'similarity_score': similar_scores
    })



## 7. Example Usage

In [8]:
# Generate recommendations for a sample user
sample_user = train_matrix.index[0]

print(f"Top 5 recommendations for User {sample_user}:")
recs = get_recommendations(sample_user, n_recommendations=5)
print(recs)

print(f"\n\nMost similar users to User {sample_user}:")

similar = find_similar_users(sample_user, n_similar=5)
print(similar)

Top 5 recommendations for User 1:
  clean_title  predicted_rating  vote_average
0  persuasion               5.0         6.000
1  persuasion               5.0         7.300
2  persuasion               5.0         7.064
3  persuasion               5.0         0.000
4    lamerica               5.0         7.200


Most similar users to User 1:
   user_id  similarity_score
0      171          0.136492
1      329          0.133222
2       49          0.129723
3      494          0.110894
4      347          0.109595


## 8. Model Summary & Performance

In [9]:

print(f"\nMODEL PERFORMANCE:")
print(f"  • RMSE: {rmse:.4f}")
print(f"  • MAE: {mae:.4f}")
print(f"  • Explained Variance: {svd_model.explained_variance_ratio_.sum():.4%}")
print(f"  • Test samples evaluated: {len(test_predictions):,}")




MODEL PERFORMANCE:
  • RMSE: 0.3256
  • MAE: 0.0877
  • Explained Variance: 99.9932%
  • Test samples evaluated: 63,858


## 10. Content-Based Filtering Model

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix



# Use movie overview/description for content vectorization
# Fill NaN values with empty strings
movie_content = df.groupby('movieid').first()['overview'].fillna('')

print(f"\nMovies with content: {len(movie_content)}")
print(f"Sample overview length: {len(movie_content.iloc[0])} characters")

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=500,           # Limit to 500 most important features
    stop_words='english',       # Remove common English words
    min_df=2,                   # Minimum document frequency
    max_df=0.8,                 # Maximum document frequency (remove very common terms)
    ngram_range=(1, 2)          # Use unigrams and bigrams
)

# Fit and transform the content
content_matrix = tfidf.fit_transform(movie_content)

print(f"TF-IDF matrix shape: {content_matrix.shape}")



Movies with content: 6834
Sample overview length: 303 characters
TF-IDF matrix shape: (6834, 500)


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute content-based similarity matrix
print("\nComputing content-based similarity matrix...")
content_similarity = cosine_similarity(content_matrix, content_matrix)

print(f"Content similarity matrix shape: {content_similarity.shape}")
print(f"Similarity range: [{content_similarity.min():.4f}, {content_similarity.max():.4f}]")

# Create a mapping from movieid to index
movieid_to_idx = {movieid: idx for idx, movieid in enumerate(movie_content.index)}
idx_to_movieid = {idx: movieid for movieid, idx in movieid_to_idx.items()}




Computing content-based similarity matrix...
Content similarity matrix shape: (6834, 6834)
Similarity range: [0.0000, 1.0000]


In [12]:
def get_content_based_recommendations(movie_id, n_recommendations=5):
    """
    Generate content-based recommendations similar to a given movie.
    
    Uses TF-IDF vectorization and cosine similarity on movie descriptions.
    
    Parameters:
    - movie_id: The ID of the reference movie
    - n_recommendations: Number of similar movies to return
    
    Returns:
    - DataFrame with recommended movies, titles, and similarity scores
    """
    if movie_id not in movieid_to_idx:
        return f"Movie {movie_id} not found in dataset"
    
    # Get the index of the movie
    movie_idx = movieid_to_idx[movie_id]
    
    # Get similarity scores with all other movies
    similarity_scores = content_similarity[movie_idx]
    
    # Get top similar movies (excluding the movie itself)
    similar_indices = np.argsort(similarity_scores)[::-1][1:n_recommendations+1]
    similar_movie_ids = [idx_to_movieid[idx] for idx in similar_indices]
    similar_scores = similarity_scores[similar_indices]
    
    # Get movie details
    similar_movies = df[df['movieid'].isin(similar_movie_ids)][
        ['movieid', 'clean_title', 'vote_average', 'release_year']
    ].drop_duplicates(subset=['movieid'])
    
    # Add similarity scores
    similar_movies['content_similarity'] = similar_movies['movieid'].map(
        {similar_movie_ids[i]: similar_scores[i] for i in range(len(similar_movie_ids))}
    )
    
    # Sort by similarity score
    similar_movies = similar_movies.sort_values('content_similarity', ascending=False)
    
    return similar_movies[['clean_title', 'content_similarity', 'vote_average', 'release_year']].head(n_recommendations)



## 11. Content-Based Recommendations Example

In [13]:
# Example: Get top 5 content-based recommendations for a movie
# First, get a sample movie from the dataset
sample_movie_id = df['movieid'].iloc[10]
sample_movie_title = df[df['movieid'] == sample_movie_id]['clean_title'].iloc[0]

print(f"Reference Movie: {sample_movie_title} (ID: {sample_movie_id})")
print("\nTop 5 Content-Based Similar Movies:")

content_recs = get_content_based_recommendations(sample_movie_id, n_recommendations=5)
print(content_recs.to_string())

# Show another example with a different movie
sample_movie_id_2 = df['movieid'].iloc[50]
sample_movie_title_2 = df[df['movieid'] == sample_movie_id_2]['clean_title'].iloc[0]

print(f"\nReference Movie: {sample_movie_title_2} (ID: {sample_movie_id_2})")
print("\nTop 5 Content-Based Similar Movies:")


content_recs_2 = get_content_based_recommendations(sample_movie_id_2, n_recommendations=5)
print(content_recs_2.to_string())

Reference Movie: heat (ID: 6)

Top 5 Content-Based Similar Movies:
         clean_title  content_similarity  vote_average  release_year
49983  inherent vice            0.451972         6.577          2014
20484         friday            0.446002         7.148          1995
57935           dope            0.419223         7.092          2015
19462    next friday            0.416701         6.431          2000
57032          chips            0.413426         6.127          2017

Reference Movie: clerks (ID: 223)

Top 5 Content-Based Similar Movies:
             clean_title  content_similarity  vote_average  release_year
250739     daddys home 2            0.420751         6.270          2017
35781         murderball            0.413390         7.036          2005
19723          mannequin            0.363542         6.823          1987
53122   love and pigeons            0.363006         7.665          1984
236959           college            0.360691         4.700          2008


### 3. Hybrid Recommendation Model (SVD + Content-Based)

A hybrid recommendation system combines:

Collaborative Filtering (SVD) → learns from user ratings
Content-Based Filtering (TF-IDF + Cosine Similarity) → uses movie features

This helps solve:

❄️ Cold start problem (new users / movies)
📈 Better accuracy than single models
🎯 More personalized recommendations
🧠 Idea of Hybrid Model

We combine:

1. SVD Prediction Score

→ “What the user is likely to rate this movie”

2. Content Similarity Score

→ “How similar the movie is to what the user likes”

In [14]:
def hybrid_recommendations(user_id, movie_id=None, n=5, alpha=0.7):
    
    if user_id not in train_matrix.index:
        return f"User {user_id} not found"

    user_rated = train_matrix.loc[user_id]
    user_rated_movies = user_rated[user_rated > 0].index.tolist()

    hybrid_scores = []

    for movie in train_matrix.columns:
        if movie in user_rated_movies:
            continue

        collab_score = predict_rating(user_id, movie)
        if collab_score is None:
            collab_score = global_mean

        content_score = 0

        if movie_id is not None and movie_id in movieid_to_idx and movie in movieid_to_idx:
            idx1 = movieid_to_idx[movie_id]
            idx2 = movieid_to_idx[movie]
            content_score = content_similarity[idx1, idx2]

        collab_score_norm = collab_score / 5.0
        content_score_norm = content_score

        final_score = alpha * collab_score_norm + (1 - alpha) * content_score_norm

        hybrid_scores.append({
            "movieid": movie,
            "score": final_score
        })

    hybrid_df = pd.DataFrame(hybrid_scores)

    # keep best score per movie
    hybrid_df = hybrid_df.groupby("movieid", as_index=False)["score"].max()

    # clean movie lookup (1 row per movie)
    movie_lookup = df.groupby("movieid", as_index=False).agg({
        "clean_title": "first",
        "vote_average": "mean"
    })

    # merge
    result = hybrid_df.merge(movie_lookup, on="movieid", how="left")

    # FINAL CLEANING: ensure unique titles + top N only
    result = result.drop_duplicates(subset=["clean_title"])
    result = result.sort_values("score", ascending=False).head(n)

    return result[["clean_title", "score", "vote_average"]]

In [15]:
sample_user = train_matrix.index[0]
sample_movie = df['movieid'].iloc[10]

hybrid_results = hybrid_recommendations(
    user_id=sample_user,
    movie_id=sample_movie,   # optional but improves quality
    n=5,
    alpha=0.7                # 70% collaborative, 30% content
)

print(hybrid_results)

                               clean_title     score  vote_average
5700  kung fu panda secrets of the masters  0.802354      6.428000
5767                             tangerine  0.798184      2.684091
3527                               nirvana  0.784229      1.461067
6169                                 chips  0.784125      3.257333
324                           blade runner  0.783826      7.931000
